# 🔮 Nexora — Supervised Purchase & Reorder Prediction
**Phase 7: Binary Classification & Precision-Recall Optimization**

### Objective
Predict whether a customer will reorder a specific product in their next purchase instance:
- Strict temporal user-cohort split (80% train / 20% validation) preventing data leakage.
- Benchmark explainable baseline (**Logistic Regression**) against **Random Forest** and **LightGBM**.
- In-depth evaluation via ROC-AUC, PR-AUC, Confusion Matrix, and Precision-Recall decision thresholds for business campaign targeting.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")
DATA_DIR = Path("../data/processed")

with open(DATA_DIR / "model_evaluation_metrics.json", "r") as f:
    metrics_data = json.load(f)

df_models = pd.DataFrame(metrics_data["models_benchmark"])
print("=== Supervised Model Benchmark Results ===")
display(df_models[["model", "roc_auc", "pr_auc", "precision", "recall", "f1_score"]])

---
## ❓ Business Question 1: How does model complexity improve discriminative power (ROC-AUC & PR-AUC)?
*Compare Logistic Regression Baseline vs. Random Forest vs. LightGBM.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC-AUC Comparison
sns.barplot(data=df_models, x="model", y="roc_auc", ax=axes[0], palette="Blues_d")
axes[0].set_title("ROC-AUC Comparison Across Models", fontsize=13, fontweight="bold")
axes[0].set_ylim(0.6, 0.90)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 7), textcoords='offset points', fontweight='bold')

# PR-AUC Comparison
sns.barplot(data=df_models, x="model", y="pr_auc", ax=axes[1], palette="Reds_d")
axes[1].set_title("PR-AUC (Average Precision) Comparison", fontsize=13, fontweight="bold")
axes[1].set_ylim(0.2, 0.55)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 7), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

---
## ❓ Business Question 2: What are the strongest predictive signals driving repurchase decisions?
*Examine feature importances from the promoted LightGBM model.*

In [ ]:
fi = pd.DataFrame(list(metrics_data["feature_importances"].items()), columns=["feature", "importance"]).sort_values(by="importance", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=fi, y="feature", x="importance", palette="viridis")
plt.title("LightGBM Feature Importances (Purchase Prediction)", fontsize=14, fontweight="bold")
plt.xlabel("Feature Importance (Gain / Splits)")
plt.ylabel("Feature")
plt.show()

---
## ❓ Business Question 3: How should business decision thresholds be tuned for different campaigns?
*Precision vs. Recall trade-off curve across classification thresholds (0.10 to 0.50).* 

In [ ]:
df_th = pd.DataFrame(metrics_data["threshold_analysis"])

plt.figure(figsize=(11, 5))
plt.plot(df_th["threshold"], df_th["precision"], marker="o", label="Precision (Quality of Recommendations)", color="#1d3557", linewidth=2.5)
plt.plot(df_th["threshold"], df_th["recall"], marker="s", label="Recall (Coverage of Reorders)", color="#e63946", linewidth=2.5)
plt.plot(df_th["threshold"], df_th["f1_score"], marker="^", label="F1-Score (Balanced)", color="#2a9d8f", linewidth=2.5, linestyle="--")

plt.title("Business Decision Threshold Curve (Precision vs. Recall Trade-Off)", fontsize=14, fontweight="bold")
plt.xlabel("Probability Threshold")
plt.ylabel("Metric Score")
plt.axvline(x=0.20, color="purple", linestyle=":", label="Recommended Default Threshold (0.20)")
plt.legend()
plt.show()